## DATA2-234
### Steps
* Re-run DExposure table from minimum Monza business date (2024-09-23)
* Re-run Loan table

## AWS-153

In [0]:
%sql
--toolcore
CREATE OR REPLACE TABLE `p-cdh`.derivativeactive.dportfoliotoolcore_liquid
TBLPROPERTIES (
            'delta.logRetentionDuration' = 'interval 5 weeks',
            'delta.deletedFileRetentionDuration' = 'interval 5 weeks',
            'delta.minReaderVersion' = '3',
            'delta.minWriterVersion' = '7',
            'delta.columnMapping.mode' = 'name',
            'delta.dataSkippingStatsColumns' = 'BusinessDate'
        )
as select * from `p-cdh`.derivativeactive.dportfoliotoolcore 

In [0]:
%sql
ALTER TABLE `p-cdh`.derivativeactive.dportfoliotoolcore_liquid
CLUSTER BY (BusinessDate)

In [0]:
%sql
OPTIMIZE  `p-cdh`.derivativeactive.dportfoliotoolcore_liquid FULL

In [0]:
%sql
ALTER TABLE `p-cdh`.derivativeactive.dportfoliotoolcore RENAME TO `p-cdh`.derivativeactive.dportfoliotoolcore_backup

In [0]:
%sql
ALTER TABLE `p-cdh`.derivativeactive.dportfoliotoolcore_liquid RENAME TO `p-cdh`.derivativeactive.dportfoliotoolcore

In [0]:
%sql
---securitized
CREATE OR REPLACE TABLE `p-cdh`.derivativeactive.dportfoliotoolsecuritized_liquid
TBLPROPERTIES (
            'delta.logRetentionDuration' = 'interval 5 weeks',
            'delta.deletedFileRetentionDuration' = 'interval 5 weeks',
            'delta.minReaderVersion' = '3',
            'delta.minWriterVersion' = '7',
            'delta.columnMapping.mode' = 'name',
            'delta.dataSkippingStatsColumns' = 'BusinessDate'
        )
as select * from `p-cdh`.derivativeactive.dportfoliotoolsecuritized

In [0]:
%sql
ALTER TABLE `p-cdh`.derivativeactive.dportfoliotoolsecuritized_liquid
CLUSTER BY (BusinessDate)

In [0]:
%sql
OPTIMIZE  `p-cdh`.derivativeactive.dportfoliotoolsecuritized_liquid FULL

In [0]:
%sql
ALTER TABLE `p-cdh`.derivativeactive.dportfoliotoolsecuritized RENAME TO `p-cdh`.derivativeactive.dportfoliotoolsecuritized_backup

In [0]:
%sql
ALTER TABLE `p-cdh`.derivativeactive.dportfoliotoolsecuritized_liquid RENAME TO `p-cdh`.derivativeactive.dportfoliotoolsecuritized

## DBTC-1407

In [0]:
%sql
DELETE FROM `p-cdh`.primaryarchive.assignment
WHERE SourceSystem = 'qc'

In [0]:
%sql
INSERT INTO `p-cdh`.primaryarchive.assignment
SELECT * FROM `d-cdh`.reference.assignment_archive_qc_test

## DBTC-1427

In [0]:
%sql
CREATE OR REPLACE TABLE `p-cdh`.primaryarchive.lg
AS
SELECT * FROM `qa-cdh`.reference.lg_hist_test

In [0]:
%sql
CREATE OR REPLACE VIEW `p-cdh`.primary.lg
AS
SELECT
  *
FROM
  `p-cdh`.primaryarchive.lg
WHERE
  BusinessDate < (
    SELECT
      MIN(BusinessDate) as start_date
    FROM
      `p-cdh`.primaryactive.lg
  )
UNION ALL
SELECT
  *
FROM
`p-cdh`.primaryactive.lg

## Refresh views

In [0]:
views_df = spark.sql("""
    SELECT table_name, table_schema, table_catalog
    FROM system.information_schema.tables
    WHERE table_catalog in ('p-cdh')
      AND table_schema in ('derivative', 'primary')
      and table_type='VIEW'
""")

# display(views_df)

for view_row in views_df.collect():
    table_name = view_row["table_name"]
    table_schema = view_row["table_schema"]
    table_catalog = view_row["table_catalog"]

    # print(table_name)

    full_view_name =f"`{table_catalog}`.{table_schema}.{table_name}"

    describe_full_view_name = spark.sql(f"""
              DESCRIBE EXTENDED {full_view_name}""")
    
    describe_full_view_name.createOrReplaceTempView("describe_full_view_name")

   # print(describe_full_view_name)

    describe_query = spark.sql(f"""
              select data_type--REGEXP_REPLACE(TRIM(data_type), '[^a-zA-Z0-9*<> ]', '') 
              AS cleaned_view_text 
              from describe_full_view_name 
              where col_name ='View Text'
              """)
    
    sql_view_creation = describe_query.collect()[0][0]

    create_or_replace_sql = f"""
          create or replace view {full_view_name} as 
          {sql_view_creation}
          """
    
    spark.sql(create_or_replace_sql)



## DBTC-1324
Closed loans function update

In [0]:
env="p"

In [0]:
# Get the schema of the dexposure table
dexposure_table = spark.table(f"`{env}-cdh`.derivative.dexposure")
dexposure_schema_fields = dexposure_table.schema.fields


# Convert the schema fields into the required string format
dexposure_schema = ",\n".join([
    f"{field.name} {field.dataType.simpleString()}"
    for field in dexposure_schema_fields
])


spark.sql(f'''
CREATE OR REPLACE FUNCTION `{env}-cdh`.derivative.get_all_exposures_by_business_date(
    business_date Date
)
RETURNS TABLE 
( 
{dexposure_schema}
)
RETURN

SELECT B.*
FROM(
        SELECT  ExposureKey, MAX(BusinessDate) BusinessDate
        FROM `p-cdh`.derivative.dexposure
        WHERE BusinessDate <= business_date
        GROUP BY ExposureKey
) A
INNER JOIN   `{env}-cdh`.derivative.dexposure B
on a.ExposureKey = b.ExposureKey
and a.BusinessDate = b.BusinessDate
AND B.IsActiveFlagInt = 0
AND B.JustificationCode != 43023

UNION

SELECT * FROM `{env}-cdh`.derivative.dexposure
WHERE BusinessDate = business_date
AND IsActiveFlagInt = 1
''')